<a href="https://colab.research.google.com/github/amadisamantha-arch/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/amadisamantha-arch/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

get_ipython().system('pip install -q duckdb huggingface_hub pandas')

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("Working dir:", os.getcwd())
print("Token loaded:", "HF_TOKEN" in os.environ)

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship
Token loaded: True


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':      f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_month':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily_month          9,841,378 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content item (a page), identified by content_hash_id.
Time window: mid-panel month 2026-03, using fact_content_daily_performance
for daily performance, joined to dim_content for page-level metadata.

I'm using March 2026 (not the sealed June sample) to avoid building label
logic on the final outcome window, per the assignment's warning.

In [ ]:
grain_check = con.sql(f"""
    SELECT content_hash_id, COUNT(*) as row_count
    FROM {TABLES['dim_content']}
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
""").df()
print(f"Content items with duplicate rows in dim_content: {len(grain_check)}")
print("If 0, content_hash_id is confirmed as the grain for dim_content.")

Content items with duplicate rows in dim_content: 0
If 0, content_hash_id is confirmed as the grain for dim_content.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields (knowable before the decision point, safe to use as model inputs):
- gsc_impressions, gsc_clicks, gsc_avg_position (from fact_content_daily_performance)
- word_count, content_type, main_intent, search_volume, competition_level (from dim_content)
- content_created_date, content_updated_date -> used to compute freshness/age

Label / proxy field:
- A decline label built from gsc_impressions trend across the month
  (e.g. did impressions drop month-over-month) — this is what I'd rank pages by.

Context fields (useful for grouping/joining, not model inputs):
- client_hash_id, content_hash_id, keyword_hash_id, url_hash_id
- client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available

Excluded fields (and why):
- keyword_char_count, url_char_count, keyword_token_count: raw-origin-adjacent
  metadata not clearly tied to performance; low direct relevance to this lane
- provider_used, model_used: describe content production process, not
  performance signal — out of scope for a refresh-scoring lane
- backlinks, category_count: not verified as populated/reliable in this slice;
  would need a dedicated check before trusting as a feature
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other:
  excluded per the data guide's warning that AI-session data is extremely
  sparse (30,177 of 78.8M rows) — including it risks a misleading signal on
  this small slice

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
fact_grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) as row_count
    FROM {TABLES['fact_daily_month']}
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print(f"Duplicate (content, date) pairs: {len(fact_grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (content, date) pairs: 0


In [ ]:
span = con.sql(f"""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as earliest,
           MAX(report_date) as latest,
           COUNT(DISTINCT content_hash_id) as unique_pages,
           COUNT(DISTINCT client_hash_id) as unique_clients
    FROM {TABLES['fact_daily_month']}
""").df()
print(span)

   row_count   earliest     latest  unique_pages  unique_clients
0    9841378 2026-03-01 2026-03-31        331437              55


In [ ]:
availability = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as gsc_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as ga4_available_rows
    FROM {TABLES['fact_daily_month']}
""").df()
print(availability)
print(f"\n{availability['gsc_available_rows'][0]} of {availability['total_rows'][0]} rows have GSC data available.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  ga4_available_rows
0     9841378             3611061              413966

3611061 of 9841378 rows have GSC data available.


Five features for Lane 2 (Refresh/Content Opportunity Scoring), each knowable
before the decision point:

1. impressions_month — total gsc_impressions for the month. Knowable because
   it's an observed measurement of what already happened, not a future outcome.
2. clicks_month — total gsc_clicks for the month. Same reasoning as above.
3. avg_position — average gsc_avg_position across the month. Observed ranking
   position, known as soon as the search data lands.
4. word_count — from dim_content, set when the page was authored/last edited.
   Known at any point after publication, well before any decision moment.
5. days_since_updated — computed as (report window end) - content_updated_date.
   Knowable because content_updated_date is a historical fact, not a future one.

In [ ]:
features = con.sql(f"""
    SELECT
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_month,
        SUM(f.gsc_clicks) AS clicks_month,
        AVG(f.gsc_avg_position) AS avg_position,
        ANY_VALUE(d.word_count) AS word_count,
        DATE_DIFF('day', ANY_VALUE(d.content_updated_date), MAX(f.report_date)) AS days_since_updated
    FROM {TABLES['fact_daily_month']} f
    JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id
    HAVING impressions_month >= 100
""").df()

print(f"Feature frame built: {len(features)} pages with enough impressions to be meaningful")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame built: 101441 pages with enough impressions to be meaningful


,content_hash_id,impressions_month,clicks_month,avg_position,word_count,days_since_updated
0,content_2e6360ad20fd7107,899.0,1.0,5.145765,2855,-90
1,content_d49a012dcb924e31,329.0,0.0,5.177774,2993,-90
2,content_614baf2af4330bd7,772.0,1.0,4.685335,3500,-90
3,content_225dc9235023be5f,488.0,1.0,17.148172,3287,-90
4,content_babcf791dccc1610,181.0,0.0,10.479603,3446,-90


The trap: add a label-derived column deliberately, watch the score jump toward
perfect, then remove it. This is notebook 02's leakage lesson, repeated here
on real warehouse data.

In [ ]:
# Quick label: did this page's impressions fall in the second half of the month
# vs the first half? (Using within-month halves since we only pulled one month.)
label_check = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily_month']}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING imp_first_half >= 50
""").df()

label_check['is_declining'] = (label_check['imp_second_half'] < 0.8 * label_check['imp_first_half']).astype(int)

model_frame = features.merge(label_check[['content_hash_id', 'is_declining', 'imp_second_half']], on='content_hash_id', how='inner')
print(f"Rows with label: {len(model_frame)}")
print(f"Decline rate: {model_frame['is_declining'].mean():.1%}")

Rows with label: 90413
Decline rate: 27.1%


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# HONEST version: only pre-decision features
honest_cols = ['impressions_month', 'clicks_month', 'avg_position', 'word_count', 'days_since_updated']
X = model_frame[honest_cols].fillna(0)
y = model_frame['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"HONEST model ROC AUC: {honest_auc:.3f}")

# LEAKY version: sneak in imp_second_half, which is literally the outcome
leaky_cols = honest_cols + ['imp_second_half']
X_leak = model_frame[leaky_cols].fillna(0)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
leaky_auc = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])
print(f"LEAKY model ROC AUC (with imp_second_half as a feature): {leaky_auc:.3f}  <- looks amazing, but it's cheating")
print("\nRemoving the leaked column and keeping only the honest score.")

HONEST model ROC AUC: 0.583
LEAKY model ROC AUC (with imp_second_half as a feature): 1.000  <- looks amazing, but it's cheating

Removing the leaked column and keeping only the honest score.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitations of this slice:

1. Unbalanced panel: only 55 of 104 total clients have data in this March 2026
   slice, and history length varies per client — some have 12+ months, others
   started tracking much more recently. Any cross-client comparison should
   account for this.

2. GSC availability is partial: only 3,611,061 of 9,841,378 rows (36.7%) have
   gsc_data_available = TRUE. GA4 availability is even thinner (4.2%). This
   means many rows in the raw fact table carry no usable search signal at all,
   and any feature built from GSC columns must filter with IS TRUE first or
   it will silently include zero-signal rows as if they were real zeros.

3. Within-month window is short: my decline label only compares the first
   half of March to the second half, which is a much shorter and noisier
   window than a full 90-day comparison. This is a limitation of doing a
   quick Week 3 exercise on one month; the actual capstone should use a
   longer, non-overlapping feature/target window.

4. GSC-only early rows: per the lane guide, rows before a client's GA4 start
   date will have ga4_data_available = FALSE even if GSC data exists, so
   "no engagement signal" must not be read as "no traffic" — it may just mean
   analytics tracking hadn't started yet for that client.

- Five plain-words contract answers: done in Section 1/2 ✓
- Three verification queries with visible output, IS TRUE used for availability ✓
- Five-feature frame with "available when?" reasoning per feature ✓
- Deliberate-leak experiment shown, then honest score kept ✓
- One named limitation of the slice (four given) ✓

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.